# SmartRoute-OSM: Algoritmo de Dijkstra

Neste segundo notebook do pipeline, carregaremos o arquivo `.graphml` salvo na etapa anterior. Em seguida, implementaremos o **Algoritmo de Dijkstra** para encontrar o menor caminho (com base na distância em metros) entre dois pontos na malha viária de Quixadá.

In [6]:
import os
import time
import heapq
import osmnx as ox
import pandas as pd

# Carregar o grafo pré-processado
data_path = "../data/quixada_drive.graphml"

if os.path.exists(data_path):
    G = ox.load_graphml(data_path)
    print(f"Grafo carregado com sucesso! Nós: {len(G.nodes)}, Arestas: {len(G.edges)}")
else:
    raise FileNotFoundError("Arquivo 'quixada_drive.graphml' não encontrado. Execute o Notebook 01 primeiro.")

Grafo carregado com sucesso! Nós: 3647, Arestas: 9762


## 1. Definição do Par Origem-Destino
Escolhemos coordenadas de latitude e longitude na cidade e identificamos os nós da malha viária mais próximos a elas.

In [7]:
# Coordenadas fixas de exemplo em Quixadá
origem_coords = (-4.9685, -39.0161)
destino_coords = (-4.9780, -39.0050)

# Mapear coordenadas para os nós do grafo
origem_node = ox.distance.nearest_nodes(G, X=origem_coords[1], Y=origem_coords[0])
destino_node = ox.distance.nearest_nodes(G, X=destino_coords[1], Y=destino_coords[0])

print(f"ID Nó Origem: {origem_node}")
print(f"ID Nó Destino: {destino_node}")

ID Nó Origem: 252615233
ID Nó Destino: 4829461035


## 2. Implementação do Algoritmo de Dijkstra
Abaixo implementamos o algoritmo clássico de Dijkstra utilizando uma fila de prioridade (min-heap) para garantir eficiência $O((V + E) \log V)$.

In [8]:
def dijkstra_routing(graph, start_node, target_node, weight_attribute='length'):
    """
    Calcula o menor caminho em um grafo usando o algoritmo de Dijkstra.
    
    Retorna:
        path (list): Lista de nós representando o caminho.
        total_cost (float): Distância/peso total percorrido.
        nodes_visited (int): Quantidade de nós explorados durante a busca.
    """
    distances = {node: float('inf') for node in graph.nodes}
    distances[start_node] = 0
    predecessors = {node: None for node in graph.nodes}
    
    priority_queue = [(0, start_node)]
    nodes_visited = 0

    while priority_queue:
        current_dist, current_node = heapq.heappop(priority_queue)
        nodes_visited += 1

        if current_node == target_node:
            break

        if current_dist > distances[current_node]:
            continue

        for neighbor in graph.neighbors(current_node):
            # Obtém o peso da primeira aresta entre os dois nós
            edge_data = graph.get_edge_data(current_node, neighbor)[0]
            edge_weight = float(edge_data.get(weight_attribute, 1))
            
            distance = current_dist + edge_weight

            if distance < distances[neighbor]:
                distances[neighbor] = distance
                predecessors[neighbor] = current_node
                heapq.heappush(priority_queue, (distance, neighbor))

    # Reconstrução do caminho
    path = []
    curr = target_node
    while curr is not None:
        path.append(curr)
        curr = predecessors[curr]
    path.reverse()

    return path, distances[target_node], nodes_visited

## 3. Execução e Medição de Desempenho
Executamos o algoritmo medindo o tempo total de processamento em milissegundos.

In [9]:
start_time = time.time()
dijkstra_path, dijkstra_dist, dijkstra_visited = dijkstra_routing(G, origem_node, destino_node, weight_attribute='length')
execution_time_ms = (time.time() - start_time) * 1000

print(f"=== RESULTADOS DIJKSTRA ===")
print(f"Distância Total: {dijkstra_dist:.2f} metros")
print(f"Nós Visitados: {dijkstra_visited}")
print(f"Tempo de Execução: {execution_time_ms:.2f} ms")

=== RESULTADOS DIJKSTRA ===
Distância Total: 1890.27 metros
Nós Visitados: 961
Tempo de Execução: 8.17 ms


## 4. Salvando Resultados Intermediários
Salvamos as métricas em formato CSV para consolidar no notebook final de benchmark (`04_benchmark_and_folium_map.ipynb`).

In [10]:
metrics_data = {
    'algorithm': ['Dijkstra'],
    'distance_m': [dijkstra_dist],
    'visited_nodes': [dijkstra_visited],
    'execution_time_ms': [execution_time_ms],
    'origem_node': [origem_node],
    'destino_node': [destino_node]
}

df_dijkstra = pd.DataFrame(metrics_data)
df_dijkstra.to_csv("../data/dijkstra_metrics.csv", index=False)

print("Métricas salvas em '../data/dijkstra_metrics.csv' com sucesso!")

Métricas salvas em '../data/dijkstra_metrics.csv' com sucesso!


In [11]:
# Compute visited nodes for Dijkstra and save
def dijkstra_visited_nodes(graph, start_node, target_node, weight_attribute='length'):
    distances = {node: float('inf') for node in graph.nodes}
    distances[start_node] = 0
    predecessors = {node: None for node in graph.nodes}
    priority_queue = [(0, start_node)]
    visited = []
    while priority_queue:
        _, current = heapq.heappop(priority_queue)
        visited.append(current)
        if current == target_node:
            break
        for neighbor in graph.neighbors(current):
            edge_data = graph.get_edge_data(current, neighbor)[0]
            w = float(edge_data.get(weight_attribute,1))
            ndist = distances[current] + w
            if ndist < distances[neighbor]:
                distances[neighbor] = ndist
                predecessors[neighbor] = current
                heapq.heappush(priority_queue,(ndist,neighbor))
    return visited

dijkstra_visited = dijkstra_visited_nodes(G, origem_node, destino_node)
import json
with open('../data/dijkstra_visited_nodes.json','w') as f:
    json.dump(dijkstra_visited, f)
